In [1]:
from NewSemantics.petri_net_processor import read_petri_net
from NewSemantics.istar_processor import read_istar_model
from NewSemantics.transition_system import combine_goal_model_and_petri_net
from tests.utilities import pretty_print,pretty_print_states

The following is the security example from the paper.

In [2]:
gm = read_istar_model("Data/example_from_paper.txt")
pn = read_petri_net("Data/demo.pnml")
# The event mapping will be computed from the names of transitions in the Petri net
lts = combine_goal_model_and_petri_net(gm, pn, event_mapping=None) 
print(f"Event map {lts.event_map}")
qualities = { q for q, _ in gm.qualities.items()}
print(f"qualities: {qualities}")
print(f"stable: {lts.check_stability(qualities)[0]}")
print(f"weakly compliant: {lts.check_weak_compliance(qualities)[0]}")


Event map {'t_11': [['AP']], 't_1': [], 't_4': [], 't_2': [['UE']], 't_5': [['G']], 't_3': [['RA']], 't_6': [['O']], 't_7': [], 't_8': [['PT']], 't_9': [], 't_10': [['FS']]}
qualities: {'DPA'}
stable: False
weakly compliant: True


This is an example, where a missing mapping makes weak compliance fail.

Instead of mapping t_6 to o (deploy OneTrust) it maps to $\epsilon$. 

Since the Petri nets gives a choice between deploying OneTrust and deploying Grafana, the system can now make a transition without executing
the corresponding task in the goal model, which leads to a path, where the quality DPA is not satisfied.

In [3]:
gm = read_istar_model("Data/example_from_paper.txt")
pn = read_petri_net("Data/demo.pnml")
pn.set_event_mapping(gm)
gm.add_event_mapping("t_6", [])
event_mapping = gm.event_mapping
lts = combine_goal_model_and_petri_net(gm, pn, event_mapping=event_mapping)
print(f"Event map {lts.event_map}")
qualities = { q for q, _ in gm.qualities.items()}
print(f"qualities: {qualities}")
print(f"stable: {lts.check_stability(qualities)[0]}")
print(f"weakly compliant: {lts.check_weak_compliance(qualities)[0]}")


Event map {'t_11': [['AP']], 't_1': [], 't_2': [['UE']], 't_4': [], 't_3': [['RA']], 't_5': [['G']], 't_6': [], 't_7': [], 't_8': [['PT']], 't_9': [], 't_10': [['FS']]}
qualities: {'DPA'}
stable: False
weakly compliant: False


In [4]:
gm = read_istar_model("Data/airline/airline_gm.txt")
pn = read_petri_net("Data/airline/airline_pn.pnml")
lts = combine_goal_model_and_petri_net(gm, pn, event_mapping=None)
print(f"Event map {lts.event_map}")
qualities = sorted({ q for q, _ in gm.qualities.items()})
print(f"qualities: {qualities}")
print(f"stable: {lts.check_stability(qualities)[0]}")
weakly_compliant, failing_states = lts.check_weak_compliance(qualities)
print(f"weakly compliant: {weakly_compliant}")
print(f"failing states: {sorted({str(m.pn_marking) for m in failing_states})}")

print("-----")

quality = qualities[0]
weakly_compliant, failing_states = lts.check_weak_compliance([quality])
print(f"quality: {quality}")
print(f"weakly compliant: {weakly_compliant}")
print(f"failing states: {sorted({str(m.pn_marking) for m in failing_states})}")

print("-----")

quality = qualities[1]
weakly_compliant, failing_states = lts.check_weak_compliance({quality})
print(f"quality: {quality}")
print(f"weakly compliant: {weakly_compliant}")
print(f"failing states: {sorted({str(m.pn_marking) for m in failing_states})}")


Event map {'t12': [['Approve Compensation']], 't11': [['Deny Compensation']], 't13': [['Document and Save File Case']], 't1': [], 't2': [["Retrieve time of arrival in the passenger's booking"]], 't5': [['Retrieve Information Internal Systems']], 't4': [['Calculate delay at arrival']], 't3': [['Retrieve time in which the doors of the aircraft were opened']], 't6': [], 't7': [['Register Judgement Criteria and type of EC']], 't8': [], 't9': [], 't10': []}
qualities: ['Delay at arrival measured appropriately', 'Extraordinary circumstances documented appropriately']
stable: True
weakly compliant: False
failing states: ['{p10}', '{p11}', '{p12}', '{p8}']
-----
quality: Delay at arrival measured appropriately
weakly compliant: True
failing states: []
-----
quality: Extraordinary circumstances documented appropriately
weakly compliant: False
failing states: ['{p10}', '{p11}', '{p12}', '{p8}']
